# 📊 Notebook 01 — Análisis Exploratorio de Datos (EDA)
## NHANES 2015-2016 · Ciclo I · Autor: Álvaro

**Objetivo:** Explorar la tabla maestra `nhanes_2015_maestra.csv` generada por el pipeline de Kedro.  
Se analizan nulos, distribuciones de variables clave y se comparan los biomarcadores de la población **Longeva (≥70 años)** vs. el resto.

**Datasets fusionados en la tabla maestra:** Demografía (DEMO_I), Presión Arterial (BPX_I), Antropometría (BMX_I), Colesterol/Glucosa (LBX), Condiciones Médicas (MCQ_I) y Tabaquismo (SMQ_I).

---

## 1. Importación de librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# ── Configuración estética ──────────────────────────────────────────────
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.15)
plt.rcParams.update({
    'figure.figsize': (14, 6),
    'axes.titleweight': 'bold',
    'axes.titlesize': 14,
})

print('✅ Librerías cargadas correctamente.')

## 2. Carga de datos desde `data/01_raw`

In [ ]:
# Ruta relativa desde la carpeta notebooks/ hacia data/01_raw/
DATA_PATH = '../data/01_raw/nhanes_2015_maestra.csv'

df = pd.read_csv(DATA_PATH)
print(f'📦 Dataset cargado: {df.shape[0]:,} filas × {df.shape[1]} columnas')
df.head()

## 2.1 ⚠️ Corrección de valores SAS missing

Los archivos `.XPT` de la CDC codifican ciertos valores faltantes como números extremadamente pequeños (`~5.4e-79`) en lugar de `NaN`.  
Antes de cualquier análisis, debemos reemplazarlos por `NaN` verdaderos.

In [ ]:
# ── Detectar y reemplazar valores SAS missing ──────────────────────────
# El formato SAS XPT codifica missing especiales como ~5.397605e-79
SAS_MISSING_THRESHOLD = 1e-70  # Cualquier valor positivo menor a esto es un missing SAS

# Contar antes
n_sas_missing = 0
for col in df.select_dtypes(include=[np.number]).columns:
    mask = (df[col] > 0) & (df[col] < SAS_MISSING_THRESHOLD)
    n_sas_missing += mask.sum()

print(f'🔍 Valores SAS missing detectados (0 < valor < 1e-70): {n_sas_missing:,}')

# Reemplazar por NaN
for col in df.select_dtypes(include=[np.number]).columns:
    mask = (df[col] > 0) & (df[col] < SAS_MISSING_THRESHOLD)
    df.loc[mask, col] = np.nan

# Verificar
n_after = 0
for col in df.select_dtypes(include=[np.number]).columns:
    mask = (df[col] > 0) & (df[col] < SAS_MISSING_THRESHOLD)
    n_after += mask.sum()
print(f'✅ Valores SAS missing restantes: {n_after}')
print(f'   Nulos totales en el dataset: {df.isnull().sum().sum():,}')

In [ ]:
# Información general del DataFrame
df.info()

In [ ]:
# Estadísticas descriptivas (primeras 20 columnas para legibilidad)
df.describe().T.round(2)

## 3. Diccionario de variables clave

| Código | Descripción | Fuente | Tipo |
|--------|-------------|--------|------|
| `SEQN` | ID único del participante | DEMO | ID |
| `RIDAGEYR` | Edad en años | DEMO | Numérica continua |
| `RIAGENDR` | Género (1=Masculino, 2=Femenino) | DEMO | Categórica |
| `RIDRETH3` | Raza/Etnia detallada | DEMO | Categórica |
| `DMDEDUC2` | Nivel educativo (adultos ≥20) | DEMO | Categórica ordinal |
| `DMDMARTL` | Estado civil | DEMO | Categórica |
| `INDFMPIR` | Ratio de ingresos/pobreza | DEMO | Numérica |
| `BPXSY1`-`BPXSY3` | Presión sistólica (lecturas 1-3) | BPX | Numérica |
| `BPXDI1`-`BPXDI3` | Presión diastólica (lecturas 1-3) | BPX | Numérica |
| `BPXPLS` | Pulso (latidos/min) | BPX | Numérica |
| `BMXWT` | Peso (kg) | BMX | Numérica |
| `BMXHT` | Talla (cm) | BMX | Numérica |
| `BMXBMI` | Índice de Masa Corporal | BMX | Numérica |
| `BMXWAIST` | Circunferencia de cintura (cm) | BMX | Numérica |
| `LBXTC` | Colesterol total (mg/dL) | LBX | Numérica |
| `LBXGLU` | Glucosa en ayunas (mg/dL) | LBX | Numérica |
| `SMQ020` | ¿Ha fumado ≥100 cigarrillos? | SMQ | Categórica |

## 4. Análisis de valores nulos

In [ ]:
# ── Tabla de nulos ─────────────────────────────────────────────────────
nulos = df.isnull().sum()
pct_nulos = (nulos / len(df) * 100).round(2)

df_nulos = pd.DataFrame({
    'Nulos': nulos,
    '% Nulos': pct_nulos
}).sort_values('% Nulos', ascending=False)

# Mostrar solo columnas que tienen al menos 1 nulo
df_nulos_filtrado = df_nulos[df_nulos['Nulos'] > 0]
print(f'🔍 Columnas con nulos: {len(df_nulos_filtrado)} de {df.shape[1]}')
df_nulos_filtrado.head(20)

In [ ]:
# ── Gráfico de barras: Top 20 columnas con más nulos ────────────────────
top_nulos = df_nulos_filtrado.head(20)

fig, ax = plt.subplots(figsize=(12, 8))
colors = sns.color_palette('flare', n_colors=len(top_nulos))
ax.barh(top_nulos.index, top_nulos['% Nulos'], color=colors)
ax.set_xlabel('Porcentaje de valores nulos (%)')
ax.set_title('Top 20 Variables con Mayor % de Nulos')
ax.invert_yaxis()

for i, (val, name) in enumerate(zip(top_nulos['% Nulos'], top_nulos.index)):
    ax.text(val + 0.5, i, f'{val:.1f}%', va='center', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# ── Mapa de calor de nulos (variables seleccionadas para legibilidad) ──
# Con 227 columnas, el heatmap completo es ilegible. Usamos un subconjunto.
cols_heatmap = [
    'RIDAGEYR', 'RIAGENDR', 'RIDRETH3', 'DMDEDUC2', 'DMDMARTL', 'INDFMPIR',
    'BPXSY1', 'BPXDI1', 'BPXSY2', 'BPXDI2', 'BPXSY3', 'BPXDI3', 'BPXPLS',
    'BMXWT', 'BMXHT', 'BMXBMI', 'BMXWAIST', 'LBXTC', 'LBXGLU',
    'SMQ020', 'MCQ010', 'MCQ160B', 'MCQ160C', 'MCQ160F'
]
cols_heatmap = [c for c in cols_heatmap if c in df.columns]

fig, ax = plt.subplots(figsize=(18, 7))
sns.heatmap(
    df[cols_heatmap].isnull().T,
    cbar=False,
    yticklabels=True,
    cmap='YlOrRd',
    ax=ax
)
ax.set_title('Mapa de Calor de Nulos — Variables Clave (NHANES 2015-2016)', fontsize=15)
ax.set_xlabel('Observaciones (pacientes)')
ax.set_ylabel('Variables')
plt.tight_layout()
plt.show()

## 5. Distribución de la variable objetivo: Edad (`RIDAGEYR`)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

edad_clean = df['RIDAGEYR'].dropna()

# Histograma
sns.histplot(edad_clean, bins=40, kde=True, color='#5B86E5', ax=axes[0])
axes[0].axvline(x=70, color='red', linestyle='--', linewidth=2, label='Umbral Longevidad (70)')
axes[0].set_title('Distribución de Edad (RIDAGEYR)')
axes[0].set_xlabel('Edad (años)')
axes[0].legend()

# Boxplot
sns.boxplot(x=edad_clean, color='#36D1DC', ax=axes[1])
axes[1].axvline(x=70, color='red', linestyle='--', linewidth=2, label='Umbral 70')
axes[1].set_title('Boxplot de Edad')
axes[1].legend()

plt.suptitle('Análisis de la Variable Edad — NHANES 2015-2016', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Estadísticas
print(f"\n📈 Edad promedio:  {edad_clean.mean():.1f} años")
print(f"📈 Mediana:        {edad_clean.median():.0f} años")
print(f"📈 Desv. estándar: {edad_clean.std():.1f}")
print(f"📈 Rango:          [{edad_clean.min():.0f}, {edad_clean.max():.0f}]")
print(f"📈 Registros con edad válida: {len(edad_clean):,} de {len(df):,}")

## 6. Creación de la variable `IS_LONGEVO` y distribución de clases

In [ ]:
# Definición: Longevo = Edad >= 70 años (solo donde la edad no sea NaN)
df['IS_LONGEVO'] = np.where(df['RIDAGEYR'].isna(), np.nan, (df['RIDAGEYR'] >= 70).astype(int))

conteo = df['IS_LONGEVO'].dropna().astype(int).value_counts()
total_valid = conteo.sum()
print('📊 Distribución de IS_LONGEVO (solo registros con edad válida):')
print(f'   No Longevo (0): {conteo.get(0, 0):,}  ({conteo.get(0, 0)/total_valid*100:.1f}%)')
print(f'   Longevo    (1): {conteo.get(1, 0):,}  ({conteo.get(1, 0)/total_valid*100:.1f}%)')
print(f'   Sin edad (NaN): {df["IS_LONGEVO"].isna().sum():,}')

fig, ax = plt.subplots(figsize=(6, 5))
colors_pie = ['#5B86E5', '#FF6B6B']
conteo.plot.pie(
    autopct='%1.1f%%',
    colors=colors_pie,
    labels=['No Longevo (< 70)', 'Longevo (≥ 70)'],
    startangle=90,
    explode=(0, 0.05),
    textprops={'fontsize': 12},
    ax=ax
)
ax.set_ylabel('')
ax.set_title('Balance de Clases — IS_LONGEVO', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Distribución de variables demográficas clave

In [ ]:
# Filtrar solo registros con edad válida para los gráficos comparativos
df_valid = df.dropna(subset=['RIDAGEYR']).copy()
df_valid['IS_LONGEVO'] = df_valid['IS_LONGEVO'].astype(int)

# ── Género ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

genero_map = {1.0: 'Masculino', 2.0: 'Femenino'}
df_valid['GENERO_LABEL'] = df_valid['RIAGENDR'].map(genero_map)

sns.countplot(data=df_valid, x='GENERO_LABEL', hue='IS_LONGEVO', palette=['#5B86E5', '#FF6B6B'], ax=axes[0])
axes[0].set_title('Distribución por Género y Longevidad')
axes[0].set_xlabel('Género')
axes[0].legend(title='Longevo', labels=['No (< 70)', 'Sí (≥ 70)'])

# ── Raza/Etnia ──────────────────────────────────────────────────────────
etnia_map = {
    1.0: 'Mex. Americano', 2.0: 'Otro Hispano', 3.0: 'Blanco NH',
    4.0: 'Negro NH', 6.0: 'Asiático NH', 7.0: 'Otro/Multirracial'
}
df_valid['ETNIA_LABEL'] = df_valid['RIDRETH3'].map(etnia_map)

sns.countplot(data=df_valid, y='ETNIA_LABEL', hue='IS_LONGEVO', palette=['#5B86E5', '#FF6B6B'], ax=axes[1])
axes[1].set_title('Distribución por Raza/Etnia y Longevidad')
axes[1].set_xlabel('Conteo')
axes[1].set_ylabel('')
axes[1].legend(title='Longevo', labels=['No (< 70)', 'Sí (≥ 70)'])

plt.tight_layout()
plt.show()

## 8. Comparación de biomarcadores: Longevos vs. No Longevos

Ahora que contamos con datos antropométricos (BMX) y de laboratorio (LBX), la comparación es mucho más rica.

In [ ]:
# ── Biomarcadores a comparar (presión + antropometría + laboratorio) ───
biomarcadores = ['BPXSY1', 'BPXDI1', 'BPXPLS', 'BMXBMI', 'BMXWAIST', 'LBXTC', 'LBXGLU', 'BMXWT']
biomarcadores = [b for b in biomarcadores if b in df_valid.columns]

n_cols = 4
n_rows = (len(biomarcadores) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 5 * n_rows))
axes = axes.flatten()

for i, var in enumerate(biomarcadores):
    sns.boxplot(
        data=df_valid, x='IS_LONGEVO', y=var,
        palette=['#5B86E5', '#FF6B6B'],
        ax=axes[i]
    )
    axes[i].set_title(f'{var}', fontsize=12, fontweight='bold')
    axes[i].set_xticklabels(['No Longevo', 'Longevo'])
    axes[i].set_xlabel('')

# Ocultar subplots vacíos
for j in range(len(biomarcadores), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Comparación de Biomarcadores: Longevos vs. No Longevos',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Tabla comparativa de medias ─────────────────────────────────────────
variables_comparar = ['RIDAGEYR', 'BPXSY1', 'BPXDI1', 'BPXPLS',
                      'BMXBMI', 'BMXWAIST', 'BMXWT', 'BMXHT',
                      'LBXTC', 'LBXGLU', 'INDFMPIR']
variables_comparar = [v for v in variables_comparar if v in df_valid.columns]

comparacion = df_valid.groupby('IS_LONGEVO')[variables_comparar].agg(['mean', 'median', 'std']).round(2)
comparacion.columns = [f'{col[0]}_{col[1]}' for col in comparacion.columns]
comparacion.index = ['No Longevo (< 70)', 'Longevo (≥ 70)']
comparacion.T

In [ ]:
# ── Violin plots para visualización detallada de la distribución ───────
vars_violin = ['BPXSY1', 'BPXDI1', 'BPXPLS', 'BMXBMI']
vars_violin = [v for v in vars_violin if v in df_valid.columns]
titles_violin = ['Presión Sistólica (1ra)', 'Presión Diastólica (1ra)', 'Pulso', 'IMC (BMI)']

fig, axes = plt.subplots(1, len(vars_violin), figsize=(5 * len(vars_violin), 6))

for i, (var, title) in enumerate(zip(vars_violin, titles_violin)):
    sns.violinplot(
        data=df_valid, x='IS_LONGEVO', y=var,
        palette=['#5B86E5', '#FF6B6B'],
        inner='quartile',
        ax=axes[i]
    )
    axes[i].set_title(title, fontsize=13, fontweight='bold')
    axes[i].set_xticklabels(['No Longevo', 'Longevo'])
    axes[i].set_xlabel('')

plt.suptitle('Distribuciones Detalladas (Violin Plots) — Longevos vs No Longevos',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 9. Matriz de correlación de variables numéricas

In [ ]:
# Seleccionar variables numéricas relevantes para la correlación
vars_corr = ['RIDAGEYR', 'RIAGENDR', 'INDFMPIR',
             'BPXSY1', 'BPXDI1', 'BPXPLS',
             'BMXWT', 'BMXHT', 'BMXBMI', 'BMXWAIST',
             'LBXTC', 'LBXGLU']
vars_corr = [v for v in vars_corr if v in df_valid.columns]

corr_matrix = df_valid[vars_corr].corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    center=0,
    square=True,
    linewidths=0.5,
    cbar_kws={'shrink': 0.8},
    ax=ax
)
ax.set_title('Matriz de Correlación — Variables Numéricas Clave', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

## 10. Análisis de Presión Arterial por Grupo Etario

In [ ]:
# Crear grupos etarios para análisis más granular
bins = [0, 18, 30, 45, 60, 70, 80, 120]
labels = ['<18', '18-29', '30-44', '45-59', '60-69', '70-79', '80+']
df_valid['GRUPO_ETARIO'] = pd.cut(df_valid['RIDAGEYR'], bins=bins, labels=labels, right=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Sistólica promedio por grupo etario
df_valid.groupby('GRUPO_ETARIO', observed=True)['BPXSY1'].mean().plot(
    kind='bar', color=sns.color_palette('viridis', n_colors=len(labels)),
    ax=axes[0], edgecolor='black', linewidth=0.5
)
axes[0].set_title('Presión Sistólica Promedio por Grupo Etario')
axes[0].set_ylabel('mmHg')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=0)

# Diastólica promedio por grupo etario
df_valid.groupby('GRUPO_ETARIO', observed=True)['BPXDI1'].mean().plot(
    kind='bar', color=sns.color_palette('magma', n_colors=len(labels)),
    ax=axes[1], edgecolor='black', linewidth=0.5
)
axes[1].set_title('Presión Diastólica Promedio por Grupo Etario')
axes[1].set_ylabel('mmHg')
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=0)

plt.suptitle('Tendencia de Presión Arterial a lo largo de la Vida',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 11. Pairplot de las variables más relevantes

In [ ]:
# Pairplot con un subset de variables clave (coloreado por longevidad)
vars_pair = ['RIDAGEYR', 'BPXSY1', 'BMXBMI', 'BPXPLS', 'IS_LONGEVO']
vars_pair = [v for v in vars_pair if v in df_valid.columns]
df_pair = df_valid[vars_pair].dropna()

g = sns.pairplot(
    df_pair,
    hue='IS_LONGEVO',
    palette=['#5B86E5', '#FF6B6B'],
    diag_kind='kde',
    plot_kws={'alpha': 0.4, 's': 15},
    diag_kws={'fill': True, 'alpha': 0.5}
)
g.figure.suptitle('Pairplot — Biomarcadores vs. Longevidad', y=1.02, fontsize=16, fontweight='bold')
# Actualizar leyenda
new_labels = ['No Longevo', 'Longevo']
for t, l in zip(g.legend.texts, new_labels):
    t.set_text(l)
plt.show()

## 12. Conclusiones del EDA

**Hallazgos clave:**

1. **Dataset rico:** La tabla maestra contiene 227 columnas de 6 datasets NHANES (Demografía, Presión Arterial, Antropometría, Laboratorio, Condiciones Médicas, Tabaquismo).
2. **Valores SAS missing:** Se detectaron ~23,895 celdas con valores codificados como `~5.4e-79` (formato SAS XPT), ahora convertidos a `NaN` real.
3. **Nulos:** Muchas variables de cuestionario (MCQ, SMQ) tienen >50% de nulos (se aplican solo a ciertos grupos). Las variables antropométricas y de presión tienen menos nulos.
4. **Desbalanceo de clases:** La proporción de Longevos (≥70) es minoritaria, lo que sugiere aplicar técnicas de balanceo o métricas adecuadas (F1, AUC-ROC).
5. **Presión arterial:** La presión sistólica muestra una tendencia creciente con la edad. La diastólica tiende a estabilizarse o disminuir en grupos >70 años.
6. **Correlaciones:** BMI correlaciona fuertemente con circunferencia de cintura y peso. La edad correlaciona positivamente con la presión sistólica.

**Siguiente paso:** Notebook 02 — Preprocesamiento (limpieza, imputación, escalado).

---
*Notebook generado como parte del pipeline de Ciencia de Datos EV3 — NHANES 2015-2016*